# Verify Security Fixes - Shell Execution

## Purpose
Verify that the critical security vulnerabilities in run_subprocess and run_command have been fixed

## Expected Results
- Dangerous commands should now be BLOCKED
- Safe commands should work
- Security errors should be raised for dangerous operations

In [1]:
import sys
sys.path.insert(0, '/Users/dheerajchand/Documents/Professional/Siege_Analytics/Code/siege_utilities')

from siege_utilities.files.shell import run_subprocess, SecurityError, ALLOWED_COMMANDS
from siege_utilities.files.operations import run_command

print("✅ Imported fixed functions")
print(f"Allowed commands: {sorted(ALLOWED_COMMANDS)}")

Importing logging from siege_utilities.core.logging
Successfully imported 15 functions from logging
Importing string_utils from siege_utilities.core.string_utils
Successfully imported 1 functions from string_utils
siege_utilities.core: Imported 16 functions


[siege_utilities] 2025-10-13 09:34:54,087 INFO: Importing hdfs_operations from siege_utilities.distributed.hdfs_operations


[siege_utilities] 2025-10-13 09:34:54,088 INFO: Successfully imported 2 functions from hdfs_operations


[siege_utilities] 2025-10-13 09:34:54,089 INFO: Importing hdfs_config from siege_utilities.distributed.hdfs_config


[siege_utilities] 2025-10-13 09:34:54,090 INFO: Successfully imported 5 functions from hdfs_config


...

[siege_utilities] 2025-10-13 09:35:00,326 INFO: DuckDB not available - using standard geospatial stack


[siege_utilities] 2025-10-13 09:35:00,574 WARNING: Could not import additional analytics utilities: cannot import name 'get_dataset_metadata' from 'siege_utilities.analytics.datadotworld_connector' (/Users/dheerajchand/Documents/Professional/Siege_Analytics/Code/siege_utilities/siege_utilities/analytics/datadotworld_connector.py)


[siege_utilities] 2025-10-13 09:35:00,575 WARNING: Could not import reporting utilities: No module named 'siege_utilities.reporting.base_template'


[siege_utilities] 2025-10-13 09:35:00,576 WARNING: Could not import chart generation functions: No module named 'siege_utilities.reporting.base_template'


INFO: Importing environment from siege_utilities.testing.environment
INFO: Successfully imported 9 functions from environment
INFO: Importing runner from siege_utilities.testing.runner
INFO: Successfully imported 6 functions from runner
✅ Imported fixed functions
Allowed commands: ['cat', 'date', 'echo', 'find', 'grep', 'head', 'hostname', 'ls', 'pwd', 'tail', 'uname', 'wc', 'which', 'whoami']


## Test 1: Safe Commands (Should Work)

In [2]:
print("\n" + "="*80)
print("TEST 1: SAFE COMMANDS (Should work)")
print("="*80)

safe_commands = [
    "pwd",
    "echo test",
    "ls -la",
    "whoami",
]

for cmd in safe_commands:
    try:
        result = run_subprocess(cmd)
        print(f"✅ {cmd}: SUCCESS")
    except Exception as e:
        print(f"❌ {cmd}: FAILED - {e}")


TEST 1: SAFE COMMANDS (Should work)
INFO: Subprocess ['pwd'] completed with return code 0. stdout: /Users/dheerajchand/Desktop/claude/siege_utilities_project/notebooks

✅ pwd: SUCCESS
INFO: Subprocess ['echo', 'test'] completed with return code 0. stdout: test

✅ echo test: SUCCESS
INFO: Subprocess ['ls', '-la'] completed with return code 0. stdout: total 3960
drwxr-xr-x@ 36 dheerajchand  staff     1152 Oct 13 09:34 .
drwxr-xr-x@ 31 dheerajchand  staff      992 Oct 13 09:26 ..
drwxr-xr-x@  2 dheerajchand  staff       64 Oct 13 08:50 \\network\share
drwxr-xr-x@  2 dheerajchand  staff       64 Oct 13 08:50 ~
-rw-r--r--@  1 dheerajchand  staff     1287 Oct 12 19:10 01_Configuration_System_Demo.ipynb
-rw-r--r--@  1 dheerajchand  staff    14299 Oct 12 19:10 02_Create_User_Client_Profiles.ipynb
drwxr-xr-x@  2 dheerajchand  staff       64 Oct 13 08:50 C:\Windows\System32
drwxr-xr-x@  2 dheerajchand  staff       64 Oct 12 23:47 client; rm -rf 
-rw-r--r--@  1 dheerajchand  staff    23083 Oct 1

## Test 2: Dangerous Commands (Should Be BLOCKED)

In [3]:
print("\n" + "="*80)
print("TEST 2: DANGEROUS COMMANDS (Should be BLOCKED)")
print("="*80)

dangerous_commands = [
    "rm -rf /",
    "cat /etc/passwd",
    "cat /etc/shadow",
    "; cat /etc/passwd",
    "&& cat /etc/passwd",
    "| cat /etc/passwd",
    "$(cat /etc/passwd)",
    "`cat /etc/passwd`",
]

blocked_count = 0
failed_count = 0

for cmd in dangerous_commands:
    try:
        result = run_subprocess(cmd)
        print(f"❌ {cmd}: EXECUTED (VULNERABILITY NOT FIXED!)")
        failed_count += 1
    except SecurityError as e:
        print(f"✅ {cmd}: BLOCKED (Security fix working)")
        blocked_count += 1
    except Exception as e:
        print(f"✅ {cmd}: BLOCKED ({type(e).__name__})")
        blocked_count += 1

print(f"\nResults: {blocked_count}/{len(dangerous_commands)} dangerous commands blocked")
if failed_count > 0:
    print(f"⚠️  WARNING: {failed_count} dangerous commands were EXECUTED!")


TEST 2: DANGEROUS COMMANDS (Should be BLOCKED)
ERROR: Security validation failed: Command 'rm' not allowed. Allowed commands: ['cat', 'date', 'echo', 'find', 'grep', 'head', 'hostname', 'ls', 'pwd', 'tail', 'uname', 'wc', 'which', 'whoami']
✅ rm -rf /: BLOCKED (Security fix working)
ERROR: Security validation failed: Access to sensitive path forbidden: /etc/passwd
✅ cat /etc/passwd: BLOCKED (Security fix working)
ERROR: Security validation failed: Access to sensitive path forbidden: /etc/shadow
✅ cat /etc/shadow: BLOCKED (Security fix working)
ERROR: Security validation failed: Command ';' not allowed. Allowed commands: ['cat', 'date', 'echo', 'find', 'grep', 'head', 'hostname', 'ls', 'pwd', 'tail', 'uname', 'wc', 'which', 'whoami']
✅ ; cat /etc/passwd: BLOCKED (Security fix working)
ERROR: Security validation failed: Command '&&' not allowed. Allowed commands: ['cat', 'date', 'echo', 'find', 'grep', 'head', 'hostname', 'ls', 'pwd', 'tail', 'uname', 'wc', 'which', 'whoami']
✅ && cat /

## Test 3: run_command Function (Should Also Be Secure)

In [4]:
print("\n" + "="*80)
print("TEST 3: run_command WITH SECURITY")
print("="*80)

# Test safe command
try:
    result = run_command("pwd")
    print(f"✅ Safe command (pwd): SUCCESS")
except Exception as e:
    print(f"❌ Safe command (pwd): FAILED - {e}")

# Test dangerous command
try:
    result = run_command("rm -rf /")
    print(f"❌ Dangerous command (rm -rf /): EXECUTED (VULNERABILITY NOT FIXED!)")
except Exception as e:
    print(f"✅ Dangerous command (rm -rf /): BLOCKED ({type(e).__name__})")

# Test command injection
try:
    result = run_command("echo test; cat /etc/passwd")
    print(f"❌ Command injection: EXECUTED (VULNERABILITY NOT FIXED!)")
except Exception as e:
    print(f"✅ Command injection: BLOCKED ({type(e).__name__})")


TEST 3: run_command WITH SECURITY
[siege_utilities] 2025-10-13 09:35:00,628 INFO: Command succeeded: pwd


✅ Safe command (pwd): SUCCESS
[siege_utilities] 2025-10-13 09:35:00,629 ERROR: Security validation failed: Command 'rm' not allowed. Allowed commands: ['cat', 'date', 'echo', 'find', 'grep', 'head', 'hostname', 'ls', 'pwd', 'tail', 'uname', 'wc', 'which', 'whoami']


[siege_utilities] 2025-10-13 09:35:00,629 ERROR: Failed to run command rm -rf /: Command 'rm' not allowed. Allowed commands: ['cat', 'date', 'echo', 'find', 'grep', 'head', 'hostname', 'ls', 'pwd', 'tail', 'uname', 'wc', 'which', 'whoami']


✅ Dangerous command (rm -rf /): BLOCKED (SecurityError)
[siege_utilities] 2025-10-13 09:35:00,629 ERROR: Security validation failed: Forbidden character ';' in command component: test;


[siege_utilities] 2025-10-13 09:35:00,629 ERROR: Failed to run command echo test; cat /etc/passwd: Forbidden character ';' in command component: test;


✅ Command injection: BLOCKED (SecurityError)


## Test 4: Custom Whitelist (Advanced Usage)

In [5]:
print("\n" + "="*80)
print("TEST 4: CUSTOM WHITELIST")
print("="*80)

# Test with custom allow list
custom_allow = {'python3', 'ls', 'echo'}

try:
    result = run_subprocess("python3 --version", allow_list=custom_allow)
    print(f"✅ Custom whitelist (python3): SUCCESS")
except Exception as e:
    print(f"Note: python3 test: {e}")

try:
    result = run_subprocess("cat /etc/passwd", allow_list=custom_allow)
    print(f"❌ Dangerous command with custom list: EXECUTED (Should be blocked!)")
except Exception as e:
    print(f"✅ Dangerous command with custom list: BLOCKED ({type(e).__name__})")


TEST 4: CUSTOM WHITELIST
INFO: Subprocess ['python3', '--version'] completed with return code 0. stdout: Python 3.11.11

✅ Custom whitelist (python3): SUCCESS
ERROR: Security validation failed: Command 'cat' not allowed. Allowed commands: ['echo', 'ls', 'python3']
✅ Dangerous command with custom list: BLOCKED (SecurityError)


## Test 5: Unsafe Mode (Backward Compatibility)

In [6]:
print("\n" + "="*80)
print("TEST 5: UNSAFE MODE (Backward Compatibility)")
print("="*80)

# Test unsafe mode with safe command
try:
    result = run_command("echo unsafe_mode_test", unsafe=True)
    print(f"✅ Unsafe mode with safe command: SUCCESS")
    print(f"   (Warning should be logged)")
except Exception as e:
    print(f"❌ Unsafe mode: FAILED - {e}")


TEST 5: UNSAFE MODE (Backward Compatibility)
[siege_utilities] 2025-10-13 09:35:00,646 WARNING: ⚠️ SECURITY WARNING: Running command without validation: echo unsafe_mode_test


[siege_utilities] 2025-10-13 09:35:00,653 INFO: Command succeeded: echo unsafe_mode_test


✅ Unsafe mode with safe command: SUCCESS
   (Warning should be logged)


## Final Summary

In [7]:
print("\n" + "="*80)
print("SECURITY FIX VERIFICATION COMPLETE")
print("="*80)
print("\nExpected Results:")
print("✅ Safe commands should execute successfully")
print("✅ Dangerous commands should be blocked with SecurityError")
print("✅ Command injection attempts should be blocked")
print("✅ Path traversal attempts should be blocked")
print("✅ Access to sensitive files should be blocked")
print("\nIf all tests show ✅, the security vulnerabilities have been fixed.")


SECURITY FIX VERIFICATION COMPLETE

Expected Results:
✅ Safe commands should execute successfully
✅ Dangerous commands should be blocked with SecurityError
✅ Command injection attempts should be blocked
✅ Path traversal attempts should be blocked
✅ Access to sensitive files should be blocked

If all tests show ✅, the security vulnerabilities have been fixed.
